# Variance scan of the Edwards–Anderson curve

This notebook is the plan. It does not run the projector.

The question is how the disorder-averaged overlap moves when the Gaussian couplings get wider, at $h = 0$ on the periodic Kagome torus. The couplings are $J_b = \sigma z_b$, with the same standard normal $z_b$ already drawn for `gaussian_nn`. The operator is

$$
H = -\sum_b J_b \sigma_i^z \sigma_j^z - \Gamma \sum_i \sigma_i^x.
$$

Rescaling gives $H(\sigma, \Gamma) = \sigma\, H(1, \Gamma/\sigma)$. Every dimensionless observable depends on the transverse field only through $\Gamma/\sigma$. The transition sits at

$$
\Gamma_c(\sigma) = \sigma\, \Gamma_c(1).
$$

Wider couplings move the drop in $q_{\mathrm{EA}}^{2}$ to larger $\Gamma$. Plotted against $\Gamma/\sigma$, the curves fall on top of each other. That collapse is the result. A failure to collapse is a bug in the scaling or in the projector. Noise added on top of a nonzero mean coupling is a different model and is not this scan.


## Observable

Two replicas share one draw of $J$ and do not interact. The equal-time overlap is

$$
q = N^{-1} \sum_i s_i^{(a)} s_i^{(b)}.
$$

The Edwards–Anderson signal is the disorder average of the pure second moment, $q_{\mathrm{EA}}^{2} = [\langle q^{2}\rangle]$. The Binder ratio is formed on each sample and then averaged,

$$
B = \frac{1}{2}\left(3 - \frac{\langle q^{4}\rangle}{\langle q^{2}\rangle^{2}}\right).
$$

The walkers carry the ground-state amplitude, so the overlap is the forward-walked tag from `kagome_qea.pqmc.project`, not a snapshot and not the one-copy energy run. Each walker starts as an independent array of $+1$ and $-1$, with shape `(2, N)`. The guiding function is $\psi = 1$. The first 40 blocks, a projection time $\tau = 8$, are discarded.

The plotted error is the jackknife error across the 32 draws. One forward-walked series is not marked resolved. A Binder crossing is a window in $\Gamma/\sigma$, not a quoted critical point. In the paramagnet, $[\langle q^{2}\rangle]$ sits near $1/N$.


## Grid

The same 32 draws are used for every width, `sample_id = 0, ..., 31`, entropy `[1000, L, sample_id, 1]`. Multiplying the stored $z_b$ by $\sigma$ changes the width without redrawing the disorder.

- Widths: $\sigma = 0.5,\ 1,\ 2$.
- Dimensionless grid: $\Gamma/\sigma = 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5$. On the $\Gamma$ axis the $\sigma = 2$ curve reaches $7$, and the $\sigma = 0.5$ curve stops at $1.75$.
- Sizes: $L = 4$ (48 sites) and $L = 6$ (108 sites), both on the full grid. $L = 8$ (192 sites) is a later curve at $\sigma = 1$ only if the $L = 6$ runs still have more than a handful of ancestral lines.
- Population: 5000 walkers, $\tau_{\mathrm{block}} = 0.2$, 40 discarded blocks, $\tau_{\mathrm{forward}} = 4$, 24 forward cycles. These settings matched the exact $L = 2$ overlap at $\sigma = 1$.


In [ ]:
import itertools

SIZES = (4, 6)
WIDTHS = (0.5, 1.0, 2.0)
RATIOS = (0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5)
N_SAMPLES = 32

jobs = list(itertools.product(SIZES, range(N_SAMPLES), WIDTHS, RATIOS))
print(f"{len(jobs)} jobs")
print(f"L=4: {sum(size == 4 for size, *_ in jobs)}    L=6: {sum(size == 6 for size, *_ in jobs)}")
for width in WIDTHS:
    gammas = tuple(width * ratio for ratio in RATIOS)
    print(f"sigma {width:<3}  Gamma {gammas}")


## Machine

The farm runs on the AutoDL box. That box has 20 CPU cores in its cgroup, about 100 GB of RAM, and an idle H800. The projector is a serial Python walker loop, so the GPU does not run it. Eighteen cores run independent jobs $(L, \mathrm{sample}, \sigma, \Gamma)$. Two cores stay free because a daily updater is already on the machine.

On the laptop, an $L = 4$ job is about five minutes at $\Gamma = 1$, and the time scales with $\Gamma$. The full $L = 4$ grid is then about 6 hours on 18 cores, and $L = 6$ is about 14 hours. The first job on the instance is timed before the rest are launched. If that core is slower than the laptop, the estimate changes and the farm still uses the same grid.

Each job is a `nohup` process with its own log, writing under `/root/autodl-tmp/kagome-qea/`. A finished file is skipped. Results are copied back to

```text
data/kagome/gaussian_nn/pbc/L{L}/pqmc/h0/sigma{sigma}/gamma{gamma}/sample{id}_seed0.npz
```

The instance stays up when the jobs finish.


## Figure

After the files are back, one notebook reads them and plots $[\langle q^{2}\rangle]$ and $[B]$ against $\Gamma$, with one line for each $\sigma$, and the same quantities against $\Gamma/\sigma$. The $\Gamma$ plot is the shift of the drop. The $\Gamma/\sigma$ plot is the collapse onto a single curve.
